## Демонстрация работы

### Без RNN

In [2]:
from pyparsing import deque
from utils.datasets import FER2013_dataset_tensor, FER2013_dataset_image
from utils.models import CalcMetrics, ResNet50_module, save_model, load_model

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.transforms import v2 as T
from torchvision.models import resnet50, ResNet50_Weights
import cv2
from ultralytics import YOLO
from utils.models import load_model
import time

transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((94, 94)),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])


label_map = {0:"angry", 1:"disgust", 2:"fear",
            3:"happy", 4:"sad", 5:"surprise", 6: "neutral"}

affectnet_labels_names =   [
    "Anger",
    "Contempt",
    "Disgust",
    "Fear",
    "Happy",
    "Neutral",
    "Sad",
    "Surprise",
  ]

len(label_map)



class FPS_counter():
    from collections import deque
    
    def __init__(self, fps_mean=5) -> None:
        self.fps_mean = fps_mean
        self.queue = deque()
        self.queue.extend([0 for i in range(self.fps_mean)])

    def __add__(self, last_time):
        """
        Let's add some magic because there was no magic for last 2 weeks :)
        """
        first_time = self.queue.popleft()
        self.queue.append(last_time)
        return 1/((last_time - first_time)/self.fps_mean)
    
    def reset(self):
        self.queue = deque()
        self.queue.extend([0 for i in range(self.fps_mean)])


e:\ML\Part2\venv\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [4]:
def Resnet_Custom_48(output_shape=7, load_path=None, to_freeze=True): # or 96 pixels
    if load_path is None:
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

        # changing input shape fror gray scale
        pretrained_conv1 = model.conv1.weight
        new_conv1 = nn.Conv2d(1, model.inplanes, kernel_size=7, stride=2, padding=3, bias=False)
        new_conv1.weight.data = pretrained_conv1.mean(dim=1, keepdim=True)
        model.conv1 = new_conv1

        # change output shape for classes
        model.fc = nn.Linear(model.fc.in_features, output_shape)

        # freeze layers for trans-learn
        if to_freeze:
            for param in model.parameters():
                param.requires_grad = False
            
            for param in model.fc.parameters():
                param.requires_grad = True

            for name, param in model.named_parameters():
                if 'conv1' in name:
                    param.requires_grad = True

    else:
        model = torch.load(load_path, weights_only=False)

    return model


In [ ]:
fps_counter = FPS_counter()
cap = cv2.VideoCapture(0)

model_detect = YOLO("models/Yolo_face_detection.pt")
model_classify = Resnet_Custom_48(output_shape=8)
# model_classify = Resnet_Custom_48()
# model_name = "Resnet13_94_shedul_weignt_true_unfreeze_1_2_3"
model_name = "Resnet14_AffectNet"
optimizer = torch.optim.AdamW(model_classify.parameters(), lr=0, weight_decay=0)

model_classify, _, _, _, _ = load_model(model_classify, 38, 0, model_name, to_load_optim=False)


model_classify.eval()

while True:
    status, img = cap.read()
    if not status:
        print('error status img')
        break


    try:
        results = model_detect(img)
        
        if results and results[0].boxes is not None:
            boxes = results[0].boxes
            if len(boxes) > 0:
                box = boxes[0]
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                face = img[y1:y2, x1:x2]

                face_tensor = transforms(face).unsqueeze(0)
                print(face_tensor.shape)
                with torch.no_grad():
                    preds = model_classify(face_tensor)[0]
                    soft_preds = nn.functional.softmax(preds, dim=0)
                    best_index = torch.argmax(soft_preds).item()
                    confidence = soft_preds[best_index].item()
                    
                    #text = f'{label_map[best_index]} - {confidence:.2f}'
                    text = f'{affectnet_labels_names[best_index]} - {confidence:.2f}'
                    
                    cv2.putText(img, text, (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    except Exception as e:
        print(f'Error: {e}')
        continue

    fps = f'{fps_counter + time.time():.2f}'
    cv2.putText(img, fps, (20, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    
    cv2.imshow('Image', img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print('exit')
        break


cap.release()
cv2.destroyAllWindows()



Loaded models/Resnet14_AffectNet_39.pt

0: 480x640 1 FACE, 16.7ms
Speed: 2.4ms preprocess, 16.7ms inference, 3.2ms postprocess per image at shape (1, 3, 480, 640)
torch.Size([1, 1, 94, 94])

0: 480x640 1 FACE, 17.1ms
Speed: 1.3ms preprocess, 17.1ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)
torch.Size([1, 1, 94, 94])

0: 480x640 1 FACE, 14.1ms
Speed: 1.3ms preprocess, 14.1ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)
torch.Size([1, 1, 94, 94])

0: 480x640 1 FACE, 12.3ms
Speed: 1.4ms preprocess, 12.3ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)
torch.Size([1, 1, 94, 94])

0: 480x640 1 FACE, 9.4ms
Speed: 1.5ms preprocess, 9.4ms inference, 2.6ms postprocess per image at shape (1, 3, 480, 640)
torch.Size([1, 1, 94, 94])

0: 480x640 1 FACE, 12.6ms
Speed: 1.7ms preprocess, 12.6ms inference, 2.6ms postprocess per image at shape (1, 3, 480, 640)
torch.Size([1, 1, 94, 94])

0: 480x640 1 FACE, 11.8ms
Speed: 2.3ms preprocess, 11.8m

### Включая RNN

In [ ]:
import cv2
from ultralytics import YOLO
from utils.models import load_model

from utils.models import Resnet_RNN_Custom

from collections import deque
from torchvision import transforms as T
import torch.nn.functional as F

frame_buffer = deque(maxlen=24)

transform = T.Compose([
    T.ToPILImage(),
    T.Grayscale(),
    T.Resize((94, 94)),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

cap = cv2.VideoCapture(0)
fps_counter = FPS_counter()

model_detect = YOLO("models/Yolo_face_detection.pt")

model_photo = Resnet_Custom_48(output_shape=8)
model_photo_name = "Resnet14_AffectNet"

model_photo, _, _, _, _ = load_model(model_photo, 38, 0, model_photo_name, to_load_optim=False)

model_classify = Resnet_RNN_Custom(model=model_photo, embedding_dim=2048, rnn_hidden=128, mode='gru')
model_name = "VIDEO_GRU_11_affectnet_attention_trial_7" # 4 7 / 5 2  / 6 10

model_classify, _, _, _, _ = load_model(model_classify, 1, 0, model_name, to_load_optim=False)

model_classify.eval()

text = ''

while True:
    try:
        status, img = cap.read()
        if not status:
            print('error status img')
            break

        results = model_detect(img)

        if results and results[0].boxes is not None:
            boxes = results[0].boxes
            if len(boxes) > 0:
                box = boxes[0]
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                face = img[y1:y2, x1:x2]

                face_tensor = transform(face)
                frame_buffer.append(face_tensor)

                if len(frame_buffer) == 24:
                    seq = torch.stack(list(frame_buffer))  # (T, C, H, W)
                    seq = seq.unsqueeze(0)      # (1, T, C, H, W)

                    with torch.no_grad():
                        preds = model_classify(seq[:, ::2])[0]
                        soft_preds = F.softmax(preds, dim=0)
                        best_index = torch.argmax(soft_preds).item()
                        confidence = soft_preds[best_index].item()

                        text = f'{label_map[best_index]} - {confidence:.2f}'
                        frame_buffer.popleft()

                cv2.putText(img, text, (x1, y1 - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
    except Exception as e:
        print(f'Error: {e}')
        continue
    
    fps = f'{fps_counter + time.time():.2f}'
    cv2.putText(img, fps, (20, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    
    cv2.imshow('Image', img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print('exit')
        break
            
cap.release()
cv2.destroyAllWindows()


Loaded models/Resnet14_AffectNet_39.pt


e:\ML\Part2\venv\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Loaded models/VIDEO_GRU_11_affectnet_attention_trial_7_2.pt

0: 480x640 1 FACE, 44.3ms
Speed: 5.9ms preprocess, 44.3ms inference, 207.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 FACE, 9.4ms
Speed: 2.9ms preprocess, 9.4ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 FACE, 9.7ms
Speed: 1.6ms preprocess, 9.7ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 FACE, 10.0ms
Speed: 1.3ms preprocess, 10.0ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 FACE, 9.5ms
Speed: 1.3ms preprocess, 9.5ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 FACE, 9.1ms
Speed: 1.2ms preprocess, 9.1ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 FACE, 8.6ms
Speed: 1.1ms preprocess, 8.6ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 FACE, 8.0ms
Speed: 1.1ms preprocess, 8.0ms inference, 1.9ms postpr